# Full agent eval: every tool, with memory

The other notebooks isolate one backend. This one is the whole agent: all 10 read-only tools bound at once, built from scratch as a LangGraph state machine (no `agent.investigate` import) and checkpointed through `agent.memory`, so questions asked on the same `thread_id` remember each other. Use it to watch tool _selection_ — which backend the model reaches for when nothing is taken away — and whether it carries context forward.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.runnables import RunnableConfig
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, MessagesState, StateGraph

from agent.llm.client import get_llm
from agent.memory import CHECKPOINTER, forget, history, thread_config, window
from agent.tools import ALL_TOOLS


### Tools and prompt

Every tool the production agent has. The prompt does not list them — each tool's own description is already sent with the schema — it only sets the ordering rules that stop the model guessing a name or a metric.


In [ ]:
TOOLS_BY_NAME = {t.name: t for t in ALL_TOOLS}
print(len(ALL_TOOLS), "tools:", ", ".join(TOOLS_BY_NAME))

SYSTEM_PROMPT = """You are Oracle, a terse infrastructure-operations assistant for the Marconi lab.
You answer questions about the lab's Docker services using ONLY the read-only
tools available to you (container status/logs, Prometheus metrics, Loki logs).

Rules:
- Call list_services first when you are unsure of an exact service name.
- Call list_metric_names / get_label_values before a PromQL query, and
  get_loki_label_values before a LogQL query. Never guess a metric or label.
- Prefer evidence over guessing: container state, then logs, then metrics.
- You are read-only. Never suggest you can restart or change anything yourself;
  recommend the action for a human operator instead.
- If the tools cannot answer, say so plainly instead of guessing.
- Be concise."""


In [ ]:
def _text(content) -> str:
    """Coerce an LLM message content (str or content-block list) to plain text."""
    if isinstance(content, str):
        return content.strip()
    return "".join(b.get("text", "") for b in content if isinstance(b, dict)).strip()


def _usage(messages: list) -> dict:
    """Total token counts reported across a batch of LLM turns."""
    total = {"input": 0, "output": 0, "total": 0}
    for m in messages:
        reported = getattr(m, "usage_metadata", None) or {}
        total["input"] += reported.get("input_tokens", 0)
        total["output"] += reported.get("output_tokens", 0)
        total["total"] += reported.get("total_tokens", 0)
    return total


### The two nodes

`model` asks the LLM what to do next, over a `window()`-trimmed view of the thread so a long conversation cannot outgrow the context. `tools` runs whatever it asked for — with 10 tools bound, the name dispatch through `TOOLS_BY_NAME` is doing real work here, unlike in the single-tool notebooks.


In [ ]:
async def call_model(state: MessagesState, config: RunnableConfig) -> dict:
    """One LLM turn, with every tool bound."""
    llm = config["configurable"]["llm"]
    prompt = [SystemMessage(content=SYSTEM_PROMPT), *window(state["messages"])]
    return {"messages": [await llm.ainvoke(prompt)]}


In [ ]:
async def call_tools(state: MessagesState) -> dict:
    """Run every tool call the model made this turn."""
    results = []
    for call in state["messages"][-1].tool_calls:
        print(f"  tool call: {call['name']}({call['args']})")
        try:
            output = await TOOLS_BY_NAME[call["name"]].ainvoke(call["args"])
        except Exception as exc:  # noqa: BLE001 - tool can raise anything
            output = f"(tool call failed: {exc})"
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
    return {"messages": results}


### The graph

`model` loops back through `tools` for as long as it keeps asking for them. Compiling with the shared `CHECKPOINTER` from `agent.memory` is what gives this notebook memory: the same checkpointer the Discord bot uses, so a `thread_id` here behaves exactly like a channel there.


In [ ]:
def next_step(state: MessagesState) -> str:
    return "tools" if state["messages"][-1].tool_calls else END


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", call_tools)
builder.add_edge(START, "model")
builder.add_conditional_edges("model", next_step, {"tools": "tools", END: END})
builder.add_edge("tools", "model")
graph = builder.compile(checkpointer=CHECKPOINTER)


### Running one question

`thread_config` supplies the thread id and the tool budget (its recursion limit); the bound LLM rides along in the same config so the nodes do not rebuild a client every turn. Tokens are tallied over only the messages this call added, so a follow-up on the same thread reports its own cost.


In [ ]:
async def ask(question: str, thread_id: str = "notebook") -> str:
    """Ask on a thread and print what the turn cost."""
    before = len(await history(thread_id))
    config = thread_config(thread_id, llm=get_llm().bind_tools(ALL_TOOLS))
    try:
        final = await graph.ainvoke(
            {"messages": [HumanMessage(content=question)]}, config
        )
        messages = final["messages"]
        answer = _text(messages[-1].content)
    except GraphRecursionError:
        messages = await history(thread_id)
        answer = "(tool budget exhausted before an answer)"

    new = messages[before:]
    usage = _usage(new)
    calls = sum(len(getattr(m, "tool_calls", []) or []) for m in new)
    print(
        f"[transcript] {len(new)} new message(s), {calls} tool call(s) - "
        f"tokens in={usage['input']} out={usage['output']} total={usage['total']}"
    )
    return answer


## A broad question

Nothing is taken away, so the model picks its own path. Watch the printed tool calls: does it start with `list_services`, and does it reach for discovery before querying Prometheus or Loki?


In [ ]:
print(await ask("Is anything wrong in the lab right now?"))


## A follow-up on the same thread

This is the memory test. The question names nothing — no container, no metric — so it is only answerable if the checkpointed thread carried the previous turn forward.


In [ ]:
print(await ask("Why? Show me the evidence for that."))


## What the thread remembers

`history()` reads the checkpoint straight back, so you can see exactly what the next turn will be working from.


In [ ]:
for m in await history("notebook"):
    label = type(m).__name__.replace("Message", "")
    body = " ".join(str(m.content).split())[:90]
    calls = [c["name"] for c in getattr(m, "tool_calls", []) or []]
    print(f"{label:9} {body}{calls or ''}")


## A clean thread

A different `thread_id` shares nothing, which is what keeps two Discord channels apart. The same question that just worked should now fall flat.


In [ ]:
print(await ask("Why? Show me the evidence for that.", thread_id="fresh"))


### Reset

`forget()` drops a thread so the notebook can be re-run from a clean slate.


In [ ]:
await forget("notebook")
await forget("fresh")
print(await history("notebook"))
